Cellule 1 : Imports et Chargement des Données

In [ ]:
# ==============================================
# CELLULE 0 : TÉLÉCHARGEMENT DES FICHIERS
# ==============================================

from google.colab import files
import os

print("="*60)
print("📂 TÉLÉCHARGEMENT DES FICHIERS")
print("="*60)
print("\n⚠️ IMPORTANT : Sélectionnez les 3 fichiers :")
print("   1. train.csv")
print("   2. test.csv")
print("   3. SampleSubmission.csv")
print("\n👉 Cliquez sur 'Choisir des fichiers' ci-dessous")

uploaded = files.upload()

# Vérification
print("\n📋 Vérification des fichiers :")
fichiers_ok = True
for file in ['train.csv', 'test.csv', 'SampleSubmission.csv']:
    if file in os.listdir():
        size = os.path.getsize(file) / 1024
        print(f"   ✅ {file} ({size:.1f} KB)")
    else:
        print(f"   ❌ {file} manquant !")
        fichiers_ok = False

if fichiers_ok:
    print("\n✅ Tous les fichiers sont prêts !")
    print("👉 Passez maintenant à la Cellule 1")
else:
    print("\n⚠️ Fichiers manquants. Veuillez les télécharger.")

📂 TÉLÉCHARGEMENT DES FICHIERS

⚠️ IMPORTANT : Sélectionnez les 3 fichiers :
   1. train.csv
   2. test.csv
   3. SampleSubmission.csv

👉 Cliquez sur 'Choisir des fichiers' ci-dessous


Saving SampleSubmission.csv to SampleSubmission (2).csv
Saving test.csv to test (2).csv
Saving train.csv to train (2).csv

📋 Vérification des fichiers :
   ✅ train.csv (338.8 KB)
   ✅ test.csv (35.4 KB)
   ✅ SampleSubmission.csv (6.4 KB)

✅ Tous les fichiers sont prêts !
👉 Passez maintenant à la Cellule 1


Partie 2 de cellule 1

In [ ]:
# ==============================================
# CELLULE 1 : IMPORTS ET CHARGEMENT
# ==============================================

from google.colab import files
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression
import warnings
from datetime import datetime
warnings.filterwarnings('ignore')

# ==============================================
# 1. INSTALLATION DES BIBLIOTHÈQUES
# ==============================================

print("📦 Installation des bibliothèques...")
!pip install lightgbm xgboost optuna catboost -q

import lightgbm as lgb
import xgboost as xgb
import optuna
from catboost import CatBoostRegressor

print("✅ Toutes les bibliothèques sont prêtes !")

# ==============================================
# 2. CHARGEMENT DES DONNÉES
# ==============================================

print("\n📂 Chargement des données...")

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
sample = pd.read_csv('SampleSubmission.csv')

print(f"Train: {train.shape}, Test: {test.shape}, Sample: {sample.shape}")

print("\n📊 Statistiques du rendement:")
print(train['Rendement_kg_ha'].describe())

# Fixer la graine
np.random.seed(42)
print("\n✅ Configuration terminée !")

📦 Installation des bibliothèques...
✅ Toutes les bibliothèques sont prêtes !

📂 Chargement des données...
Train: (1872, 15), Test: (242, 13), Sample: (242, 2)

📊 Statistiques du rendement:
count     1378.000000
mean      1867.190179
std       2673.998269
min          0.000000
25%        629.288191
50%        918.442149
75%       1379.836970
max      18582.400000
Name: Rendement_kg_ha, dtype: float64

✅ Configuration terminée !


Cellule 2 : Feature Engineering

In [ ]:
# ==============================================
# CELLULE 2 : FEATURE ENGINEERING AVEC NORMALISATION
# ==============================================

def creer_features(df, est_train=True):
    """Crée des caractéristiques essentielles avec normalisation"""
    df2 = df.copy()

    # 1. Variables cycliques
    df2['annee_cos'] = np.cos(2 * np.pi * df2['Annee_debut'] / 10)
    df2['annee_sin'] = np.sin(2 * np.pi * df2['Annee_debut'] / 10)

    # 2. Végétation
    df2['ecart_VHI'] = df2['wVHI_moyen_saison'] - df2['wVHI_min_saison']
    df2['rapport_VHI'] = df2['wVHI_min_saison'] / (df2['wVHI_moyen_saison'] + 1e-6)
    df2['produit_VHI'] = df2['wVHI_moyen_saison'] * df2['wVHI_min_saison']

    # 3. Climat
    df2['temp_precip'] = df2['Temp_moy_C'] * df2['Precip_totale_mm']
    df2['temp_humid'] = df2['Temp_moy_C'] * df2['Humidite_rel_moy_pct']
    df2['precip_humid'] = df2['Precip_totale_mm'] * df2['Humidite_rel_moy_pct']

    # 4. Normalisation par zone
    for col in ['wVHI_moyen_saison', 'wVHI_min_saison', 'Temp_moy_C',
                'Precip_totale_mm', 'Humidite_rel_moy_pct']:
        moy = df2.groupby('Zone_agro_ecologique')[col].transform('mean')
        std = df2.groupby('Zone_agro_ecologique')[col].transform('std')
        df2[f'{col}_norm'] = (df2[col] - moy) / (std + 1e-6)

    # 5. Encodage
    for col in ['Province', 'Zone_agro_ecologique', 'Categorie', 'Culture']:
        le = LabelEncoder()
        df2[col + '_enc'] = le.fit_transform(df2[col].astype(str))

    # 6. Combinaison province-culture
    df2['prov_cult'] = df2['Province'] + '_' + df2['Culture']
    le = LabelEncoder()
    df2['prov_cult_enc'] = le.fit_transform(df2['prov_cult'])

    # 7. Features avancées
    df2['precip_temp_ratio'] = df2['Precip_totale_mm'] / (df2['Temp_moy_C'] + 1)
    df2['stress_hydrique'] = df2['wVHI_min_saison'] / (df2['wVHI_moyen_saison'] + 1e-6)
    df2['vhi_temp_ratio'] = df2['wVHI_moyen_saison'] / (df2['Temp_moy_C'] + 1)

    # 8. NOUVELLES FEATURES
    df2['precip_vhi_ratio'] = df2['Precip_totale_mm'] / (df2['wVHI_moyen_saison'] + 1e-6)
    df2['temp_humid_prod'] = df2['Temp_moy_C'] * df2['Humidite_rel_moy_pct']

    # 9. Normalisation globale (StandardScaler) pour certaines features
    # Cela peut aider pour les interactions
    scaler = StandardScaler()
    cols_a_normaliser = ['precip_temp_ratio', 'stress_hydrique', 'vhi_temp_ratio',
                         'precip_vhi_ratio', 'temp_humid_prod']

    # Appliquer la normalisation
    for col in cols_a_normaliser:
        if col in df2.columns:
            # Remplacer les infinis
            df2[col] = df2[col].replace([np.inf, -np.inf], np.nan).fillna(0)
            if est_train:
                mean = df2[col].mean()
                std = df2[col].std()
                if std > 0:
                    df2[col + '_norm'] = (df2[col] - mean) / std
                else:
                    df2[col + '_norm'] = 0
            else:
                # Pour le test, utiliser les moyennes de l'entraînement (à recalculer)
                # On garde la valeur brute, le modèle s'en chargera
                pass

    # 10. Statistiques historiques (uniquement pour l'entraînement)
    if est_train:
        moy_culture = df2.groupby('Culture')['Rendement_kg_ha'].mean()
        moy_prov_cult = df2.groupby(['Province', 'Culture'])['Rendement_kg_ha'].mean()
        df2['moy_culture'] = df2['Culture'].map(moy_culture)
        df2['moy_prov_cult'] = df2.apply(
            lambda r: moy_prov_cult.get((r['Province'], r['Culture']), r['moy_culture']),
            axis=1
        )

        # Écart-type par culture (pour normaliser)
        for culture in df2['Culture'].unique():
            mask = df2['Culture'] == culture
            if mask.sum() > 3:
                std_cult = df2.loc[mask, 'Rendement_kg_ha'].std()
                df2.loc[mask, 'std_culture'] = std_cult

    return df2

print("🔧 Création des caractéristiques...")

train_feat = creer_features(train, est_train=True)
test_feat = creer_features(test, est_train=False)

print(f"✅ Train: {train_feat.shape}, Test: {test_feat.shape}")

# Colonnes à exclure
cols_exclure = ['ID', 'Rendement_kg_ha', 'Superficie_ha', 'Production_t',
                'Annee_campagne', 'prov_cult', 'Province', 'Zone_agro_ecologique',
                'Categorie', 'Culture', 'province_culture']

# Ne garder que les colonnes numériques
cols_features = [c for c in train_feat.columns if c not in cols_exclure]
cols_features = [c for c in cols_features if train_feat[c].dtype != 'object']

# Liste des features importantes (incluant les nouvelles)
cols_features_importantes = [
    'moy_prov_cult', 'prov_cult_enc', 'moy_culture', 'Culture_enc',
    'wVHI_min_saison', 'wVHI_moyen_saison', 'Province_enc',
    'produit_VHI', 'rapport_VHI', 'ecart_VHI',
    'precip_temp_ratio', 'stress_hydrique', 'vhi_temp_ratio',
    'annee_cos', 'annee_sin', 'temp_precip', 'temp_humid',
    'precip_vhi_ratio', 'temp_humid_prod',  # Nouvelles features
    'std_culture'  # Nouvelle feature
]

# Garder seulement les colonnes qui existent
cols_features = [c for c in cols_features_importantes if c in cols_features]

print(f"🔍 {len(cols_features)} caractéristiques sélectionnées")
print(f"   Features: {cols_features[:5]}...")

# Pour le test, ajouter les colonnes manquantes avec des valeurs par défaut
for col in cols_features:
    if col not in test_feat.columns:
        if col == 'std_culture':
            test_feat[col] = train_feat[col].mean() if col in train_feat.columns else 0
        else:
            test_feat[col] = 0

🔧 Création des caractéristiques...
✅ Train: (1872, 47), Test: (242, 37)
🔍 20 caractéristiques sélectionnées
   Features: ['moy_prov_cult', 'prov_cult_enc', 'moy_culture', 'Culture_enc', 'wVHI_min_saison']...


Cellule 3 : Entraînement des Modèles

In [ ]:
# ==============================================
# CELLULE 3 : ENTRAÎNEMENT AVEC CATBOOST
# ==============================================

def wmape(real, pred, poids):
    """Calcul robuste du wMAPE"""
    mask = (real > 0) & (real < 20000) & (poids > 0)
    if mask.sum() == 0:
        return np.nan

    real_f = real[mask]
    pred_f = pred[mask]
    poids_f = poids[mask]

    real_f = np.clip(real_f, 1e-6, None)
    erreur = np.abs((real_f - pred_f) / real_f)
    seuil = np.percentile(erreur, 95)
    erreur = np.clip(erreur, 0, seuil)

    return np.sum(poids_f * erreur) / np.sum(poids_f)

# ==============================================
# PRÉPARATION DES DONNÉES
# ==============================================

print("🔧 Nettoyage des données...")

# Supprimer les valeurs extrêmes
seuil_rendement = train_feat['Rendement_kg_ha'].quantile(0.99)
mask_clean = (train_feat['Rendement_kg_ha'] <= seuil_rendement) & (train_feat['Rendement_kg_ha'] >= 0)

print(f"   Avant nettoyage: {len(train_feat)} lignes")
print(f"   Après nettoyage: {mask_clean.sum()} lignes")

# Données LightGBM
X_lgb = train_feat.loc[mask_clean, cols_features].copy()
X_lgb = X_lgb.fillna(0)

# Limiter les valeurs extrêmes
for col in X_lgb.columns:
    if X_lgb[col].dtype in ['float64', 'int64']:
        mean = X_lgb[col].mean()
        std = X_lgb[col].std()
        if std > 0:
            X_lgb[col] = X_lgb[col].clip(mean - 5*std, mean + 5*std)

y_lgb = np.log1p(train_feat.loc[mask_clean, 'Rendement_kg_ha'].clip(lower=0, upper=20000))
groups_lgb = train_feat.loc[mask_clean, 'Annee_debut']

print(f"✅ Données préparées: {X_lgb.shape}")

# Splits
gkf_opt = GroupKFold(n_splits=min(3, len(X_lgb) // 30))
splits_lgb = list(gkf_opt.split(X_lgb, y_lgb, groups_lgb))
print(f"✅ {len(splits_lgb)} splits de validation")

# ==============================================
# ENTRAÎNEMENT LIGHTGBM
# ==============================================

print("\n" + "="*50)
print("ENTRAÎNEMENT LIGHTGBM")
print("="*50)

params_lgb = {
    'n_estimators': 200,
    'learning_rate': 0.05,
    'num_leaves': 20,
    'max_depth': 5,
    'min_child_samples': 20,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbose': -1
}

modeles_lgb = []
scores_lgb = []

for fold, (idx_train, idx_val) in enumerate(splits_lgb):
    print(f"Pli {fold+1}/{len(splits_lgb)}...")
    X_train, X_val = X_lgb.iloc[idx_train], X_lgb.iloc[idx_val]
    y_train, y_val = y_lgb.iloc[idx_train], y_lgb.iloc[idx_val]

    model = lgb.LGBMRegressor(**params_lgb)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])

    pred = np.expm1(model.predict(X_val))
    pred = np.maximum(pred, 0)
    real = np.expm1(y_val)

    idx_val_original = mask_clean[mask_clean].index[idx_val]
    poids = np.log1p(train_feat.loc[idx_val_original, 'Superficie_ha'] + 1)
    score = wmape(real, pred, poids)

    scores_lgb.append(score)
    modeles_lgb.append(model)
    print(f"  wMAPE: {score:.4f}")

scores_filtres = [s for s in scores_lgb if s < 10 and not np.isnan(s)]
print(f"✅ wMAPE moyen LightGBM: {np.mean(scores_filtres):.4f}")

# ==============================================
# ENTRAÎNEMENT XGBOOST
# ==============================================

print("\n" + "="*50)
print("ENTRAÎNEMENT XGBOOST")
print("="*50)

params_xgb = {
    'n_estimators': 200,
    'learning_rate': 0.05,
    'max_depth': 4,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'n_jobs': -1,
    'verbosity': 0
}

modeles_xgb = []
scores_xgb = []

for fold, (idx_train, idx_val) in enumerate(splits_lgb):
    print(f"Pli {fold+1}/{len(splits_lgb)}...")
    X_train, X_val = X_lgb.iloc[idx_train], X_lgb.iloc[idx_val]
    y_train, y_val = y_lgb.iloc[idx_train], y_lgb.iloc[idx_val]

    X_train = X_train.fillna(0).replace([np.inf, -np.inf], 0)
    X_val = X_val.fillna(0).replace([np.inf, -np.inf], 0)

    model = xgb.XGBRegressor(**params_xgb)
    model.fit(X_train, y_train)

    pred = np.expm1(model.predict(X_val))
    pred = np.maximum(pred, 0)
    real = np.expm1(y_val)

    idx_val_original = mask_clean[mask_clean].index[idx_val]
    poids = np.log1p(train_feat.loc[idx_val_original, 'Superficie_ha'] + 1)
    score = wmape(real, pred, poids)

    scores_xgb.append(score)
    modeles_xgb.append(model)
    print(f"  wMAPE: {score:.4f}")

scores_filtres_xgb = [s for s in scores_xgb if s < 10 and not np.isnan(s)]
print(f"✅ wMAPE moyen XGBoost: {np.mean(scores_filtres_xgb):.4f}")

# ==============================================
# ENTRAÎNEMENT CATBOOST (NOUVEAU)
# ==============================================

print("\n" + "="*50)
print("ENTRAÎNEMENT CATBOOST")
print("="*50)

from catboost import CatBoostRegressor

params_cat = {
    'iterations': 200,
    'learning_rate': 0.05,
    'depth': 5,
    'l2_leaf_reg': 3,
    'random_seed': 42,
    'verbose': False
}

modeles_cat = []
scores_cat = []

for fold, (idx_train, idx_val) in enumerate(splits_lgb):
    print(f"Pli {fold+1}/{len(splits_lgb)}...")
    X_train, X_val = X_lgb.iloc[idx_train], X_lgb.iloc[idx_val]
    y_train, y_val = y_lgb.iloc[idx_train], y_lgb.iloc[idx_val]

    model = CatBoostRegressor(**params_cat)
    model.fit(X_train, y_train)

    pred = np.expm1(model.predict(X_val))
    pred = np.maximum(pred, 0)
    real = np.expm1(y_val)

    idx_val_original = mask_clean[mask_clean].index[idx_val]
    poids = np.log1p(train_feat.loc[idx_val_original, 'Superficie_ha'] + 1)
    score = wmape(real, pred, poids)

    scores_cat.append(score)
    modeles_cat.append(model)
    print(f"  wMAPE: {score:.4f}")

scores_filtres_cat = [s for s in scores_cat if s < 10 and not np.isnan(s)]
if scores_filtres_cat:
    print(f"✅ wMAPE moyen CatBoost: {np.mean(scores_filtres_cat):.4f}")

print("\n✅ Modèles entraînés avec succès !")

🔧 Nettoyage des données...
   Avant nettoyage: 1872 lignes
   Après nettoyage: 1364 lignes
✅ Données préparées: (1364, 20)
✅ 3 splits de validation

ENTRAÎNEMENT LIGHTGBM
Pli 1/3...
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[96]	valid_0's l2: 0.0206728
  wMAPE: 0.0848
Pli 2/3...
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[200]	valid_0's l2: 0.23995
  wMAPE: 0.0816
Pli 3/3...
Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[71]	valid_0's l2: 0.0167612
  wMAPE: 0.0766
✅ wMAPE moyen LightGBM: 0.0810

ENTRAÎNEMENT XGBOOST
Pli 1/3...
  wMAPE: 0.0799
Pli 2/3...
  wMAPE: 0.0820
Pli 3/3...
  wMAPE: 0.0732
✅ wMAPE moyen XGBoost: 0.0784

ENTRAÎNEMENT CATBOOST
Pli 1/3...
  wMAPE: 0.0772
Pli 2/3...
  wMAPE: 0.0817
Pli 3/3...
  wMAPE: 0.0697
✅ wMAPE moyen CatBoost: 0.0762

✅ Modèles entraînés avec succès !


Cellule 4 : Prédictions et Soumission

In [ ]:
# ==============================================
# CELLULE 4 : PRÉDICTIONS AVEC 3 MODÈLES
# ==============================================

print("🔮 Génération des prédictions...")

# ==============================================
# 1. PRÉPARATION DES DONNÉES DE TEST
# ==============================================

# Ajouter les colonnes manquantes
moy_culture_train = train_feat.groupby('Culture')['Rendement_kg_ha'].mean()
moy_prov_cult_train = train_feat.groupby(['Province', 'Culture'])['Rendement_kg_ha'].mean()

test_feat['moy_culture'] = test_feat['Culture'].map(moy_culture_train)

def get_moy_prov_cult(row):
    key = (row['Province'], row['Culture'])
    if key in moy_prov_cult_train:
        return moy_prov_cult_train[key]
    else:
        return row['moy_culture']

test_feat['moy_prov_cult'] = test_feat.apply(get_moy_prov_cult, axis=1)

# Ajouter les colonnes manquantes
for col in cols_features:
    if col not in test_feat.columns:
        if col == 'std_culture':
            test_feat[col] = train_feat[col].mean() if col in train_feat.columns else 0
        else:
            test_feat[col] = 0

X_test = test_feat[cols_features].copy()
X_test = X_test.fillna(0).replace([np.inf, -np.inf], 0)

print(f"✅ Données de test: {X_test.shape}")

# ==============================================
# 2. PRÉDICTION SUR LE TEST
# ==============================================

def predire_test(modeles, X_test):
    preds = []
    for model in modeles:
        if model is not None:
            try:
                pred = np.expm1(model.predict(X_test))
                pred = np.maximum(pred, 0)
                pred = np.minimum(pred, 20000)
                preds.append(pred)
            except:
                continue
    return np.mean(preds, axis=0) if preds else np.zeros(len(X_test))

pred_lgb = predire_test(modeles_lgb, X_test)
pred_xgb = predire_test(modeles_xgb, X_test)
pred_cat = predire_test(modeles_cat, X_test)

print(f"✅ Prédictions générées")

# ==============================================
# 3. ENSEMBLE À 3 MODÈLES
# ==============================================

score_lgb = np.mean(scores_filtres) if scores_filtres else 0.5
score_xgb = np.mean(scores_filtres_xgb) if scores_filtres_xgb else 0.5
score_cat = np.mean(scores_filtres_cat) if scores_filtres_cat else 0.5

# Poids inversement proportionnels aux scores
poids_lgb = 1 / score_lgb if score_lgb > 0 else 0.5
poids_xgb = 1 / score_xgb if score_xgb > 0 else 0.5
poids_cat = 1 / score_cat if score_cat > 0 else 0.5

total = poids_lgb + poids_xgb + poids_cat
poids_lgb = poids_lgb / total
poids_xgb = poids_xgb / total
poids_cat = poids_cat / total

print(f"📊 Poids: LightGBM={poids_lgb:.3f}, XGBoost={poids_xgb:.3f}, CatBoost={poids_cat:.3f}")

pred_final = poids_lgb * pred_lgb + poids_xgb * pred_xgb + poids_cat * pred_cat

print(f"✅ Prédictions finales: {len(pred_final)} échantillons")

# ==============================================
# 4. POST-TRAITEMENT AFFINÉ
# ==============================================

print("\n🔧 Post-traitement affiné...")

# Ajustements par culture
ajustements = {
    'Riz': 1.08,
    'Manioc': 1.12,
    'Taro': 1.05,
    'Patate_douce': 0.96,
    'Arachide': 1.04,
    'Sésame': 1.04,
    'Maïs': 1.06,
    'Niébé': 1.02,
    'Sorgho': 0.98,
    'Mil': 1.01,
    'Berbéré': 1.02,
    'Blé': 1.03,
    'Fonio': 1.05
}

for culture, facteur in ajustements.items():
    mask = test_feat['Culture'] == culture
    if mask.sum() > 0:
        pred_final[mask] *= facteur

# Ajustements par province
ajustements_province = {
    'Barh El Ghazal': 0.95,
    'Kanem': 0.97,
    'Lac': 0.98,
    'Guéra': 1.03,
    'Sila': 0.97,
    'Logone Occidental': 0.98
}

for province, facteur in ajustements_province.items():
    mask = test_feat['Province'] == province
    if mask.sum() > 0:
        pred_final[mask] *= facteur

# Limites
pred_final = np.maximum(pred_final, 100)
pred_final = np.minimum(pred_final, 14000)

print(f"📊 Min: {pred_final.min():.0f}, Max: {pred_final.max():.0f}, Moyenne: {pred_final.mean():.0f}")

# ==============================================
# 5. PRÉCISION DU MODÈLE
# ==============================================

print("\n" + "="*60)
print("📊 PRÉCISION DU MODÈLE")
print("="*60)

# Calcul des métriques sur l'entraînement
pred_train_lgb = predire_test(modeles_lgb, X_lgb)
pred_train_xgb = predire_test(modeles_xgb, X_lgb)
pred_train_cat = predire_test(modeles_cat, X_lgb)
pred_train = poids_lgb * pred_train_lgb + poids_xgb * pred_train_xgb + poids_cat * pred_train_cat

real_train = np.expm1(y_lgb)
poids_train = np.log1p(train_feat.loc[mask_clean, 'Superficie_ha'] + 1)

mask_eval = (real_train > 0) & (real_train < 20000) & (poids_train > 0)
wmape_train = wmape(real_train[mask_eval], pred_train[mask_eval], poids_train[mask_eval])

print(f"\n📈 wMAPE sur l'entraînement: {wmape_train:.4f} ({wmape_train*100:.2f}%)")

# Erreur absolue moyenne
mae = np.mean(np.abs(real_train[mask_eval] - pred_train[mask_eval]))
print(f"📈 Erreur Absolue Moyenne (MAE): {mae:.2f} kg/ha")

# R²
from sklearn.metrics import r2_score
r2 = r2_score(real_train[mask_eval], pred_train[mask_eval])
print(f"📈 Coefficient de détermination (R²): {r2:.4f}")

# ==============================================
# 6. SOUMISSION
# ==============================================

print("\n📁 Création du fichier de soumission...")

submission = sample.copy()
submission['Rendement_kg_ha'] = pred_final

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
nom_fichier = f'soumission_{timestamp}.csv'
submission.to_csv(nom_fichier, index=False)

print(f"✅ Fichier: {nom_fichier}")
print(f"   Lignes: {len(submission)}")

print("\n📋 Aperçu:")
print(submission.head(10))

# ==============================================
# 7. TÉLÉCHARGEMENT
# ==============================================

try:
    from google.colab import files
    files.download(nom_fichier)
    print("\n✅ Téléchargement lancé !")
except:
    print(f"\n📁 Téléchargez manuellement: {nom_fichier}")

print("\n" + "="*60)
print("🍀 BONNE CHANCE !")
print("="*60)

🔮 Génération des prédictions...
✅ Données de test: (242, 20)
✅ Prédictions générées
📊 Poids: LightGBM=0.323, XGBoost=0.334, CatBoost=0.343
✅ Prédictions finales: 242 échantillons

🔧 Post-traitement affiné...
📊 Min: 193, Max: 9615, Moyenne: 1329

📊 PRÉCISION DU MODÈLE

📈 wMAPE sur l'entraînement: 0.0543 (5.43%)
📈 Erreur Absolue Moyenne (MAE): 121.19 kg/ha
📈 Coefficient de détermination (R²): 0.9818

📁 Création du fichier de soumission...
✅ Fichier: soumission_20260812_2136.csv
   Lignes: 242

📋 Aperçu:
                           ID  Rendement_kg_ha
0   Barh_El_Ghazal_2024_Niébé       266.992086
1    Barh_El_Ghazal_2024_Maïs       617.431328
2     Barh_El_Ghazal_2024_Mil       306.407671
3  Barh_El_Ghazal_2024_Sorgho       477.777001
4         Batha_2024_Arachide       913.002963
5            Batha_2024_Niébé       523.481058
6    Batha_2024_Pois_de_terre       569.599677
7           Batha_2024_Sésame       427.069012
8          Batha_2024_Berbéré       629.018911
9             Batha_202

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Téléchargement lancé !

🍀 BONNE CHANCE !
